

# Procesamiento de Lenguaje Natural
# Fine-tuning BERT

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,                       # Convierte texto → tokens (números)
    AutoModelForSequenceClassification,  # Modelo listo para clasificación (ej: positivo/negativo)
    TrainingArguments,                   # Configuración del entrenamiento
    Trainer                             # Clase que automatiza el entrenamiento
)

Definimos el dataset que usaremos para fine-tuning

In [ ]:
dataset = load_dataset("imdb")

# Reducimos tamaño para Colab (opcional)
dataset = dataset.shuffle(seed=42)
dataset["train"] = dataset["train"].select(range(2000))
dataset["test"] = dataset["test"].select(range(500))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Esto convierte un dataset de texto en datos numéricos listos para entrenar un modelo BERT

In [ ]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["text"],        # Texto de entrada
        truncation=True,        # Corta el texto si es muy largo
        padding="max_length",   # Rellena hasta longitud fija
        max_length=128          # Longitud máxima de tokens
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Definimos los argumentos de entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",              # Carpeta donde se guarda el modelo y checkpoints
    learning_rate=2e-5,                  # Tasa de aprendizaje (baja, típico en fine-tuning)
    per_device_train_batch_size=8,       # Batch size para entrenamiento
    per_device_eval_batch_size=8,        # Batch size para evaluación
    num_train_epochs=2,                  # Número de veces que el modelo ve todo el dataset
    weight_decay=0.01,                   # Regularización para evitar overfitting
    logging_dir="./logs"                 # Carpeta donde se guardan logs del entrenamiento
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Definimos función para calcular el accuracy

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

Definimos el trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.367724


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.3677244262695312, metrics={'train_runtime': 115.5691, 'train_samples_per_second': 34.611, 'train_steps_per_second': 4.326, 'total_flos': 263111055360000.0, 'train_loss': 0.3677244262695312, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.4676930606365204,
 'eval_runtime': 4.1208,
 'eval_samples_per_second': 121.336,
 'eval_steps_per_second': 15.288,
 'epoch': 2.0}

In [ ]:
trainer.save_model("bert-finetuned-imdb")
tokenizer.save_pretrained("bert-finetuned-imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert-finetuned-imdb/tokenizer_config.json',
 'bert-finetuned-imdb/tokenizer.json')

Seleccionamos GPU si está disponible, si no usa CPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Definimos textos de ejemplo para clasificar

In [ ]:
texts = [
    "This movie was amazing!",
    "I hated every minute of this film.",
    "It was okay, not the best.",
    "Absolutely fantastic performance!",
    "Worst movie I have ever seen."
]

Realizamos clasificaciones

In [ ]:
# Iteramos sobre cada texto
for text in texts:

    # Tokenizamos el texto, formato que entiende el modelo
    inputs = tokenizer(
        text,
        return_tensors="pt",  # devuelve tensores de PyTorch
        truncation=True,      # corta si es muy largo
        padding=True          # agrega padding si es necesario
    )

    # Movemos los tensores al mismo dispositivo que el modelo
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Inferencia (sin calcular gradientes → más rápido)
    with torch.no_grad():
        outputs = model(**inputs)

    # Extraemos las predicciones (logits)
    logits = outputs.logits

    # Elegimos la clase con mayor probabilidad
    pred = torch.argmax(logits, dim=1).item()

    # Convertimos a etiqueta humana
    label = "Positivo" if pred == 1 else "Negativo"

    # Mostramos resultado
    print(f"{text}  -->  {label}")

This movie was amazing!  -->  Positivo
I hated every minute of this film.  -->  Negativo
It was okay, not the best.  -->  Negativo
Absolutely fantastic performance!  -->  Positivo
Worst movie I have ever seen.  -->  Negativo
